# Step 4a: Preseen context notes for the field forecast

Step 4a of the Econ forecast (`../README.md`): the prompts that the Preseen question on the **field** of the 2026
Sveriges Riksbank Prize in Economic Sciences receives as context notes (API `POST /questions/{id}/context/`, one note
per file, uploaded with `../../experiment/preseen_chem30_main/nobel_preseen_exp.py add-context`). Each note cell
below **prints the full text of one note** and writes it to `context/<name>.md`; the notes are built from the data
the earlier steps completed, nothing is fetched.

**Inputs**

| File | Content |
|---|---|
| `../02_fields/results/prize_works_jel.csv` | the 64 prize works 1969–2025 with their consensus JEL code (three models, step 2) and each model's codes |
| `../data/econ_prizes_laureates.csv` | every laureate: share, official motivation, affiliation, birth and death dates (crawl of 9 October 2026) |
| `../data/classificationTree.xml` | the JEL tree (names of the codes) |
| `../01_committee/roster.yaml`, `../01_committee/profiles/*.json` | the 11 members of the 2026 committee and their merged profiles (step 1) |
| `../config.yaml` | the information cutoff date stated in every note |

**Design document.** The notes implement the design *"Forecasting the field of the 2026 prize: a design that applies
field rotation strongly"* (9 October 2026, written in Korean; rendered in English here). Every statement carries an
evidence tag: **[P]** a result reported by Dolton and Tol (2026, arXiv:2603.20767v1), **[M]** an opinion raised at
the project meeting of 9 October 2026, **[D]** a design choice of this forecast, **[O]** information confirmed from
official sources (nobelprize.org), and **[S]** (added here) a number computed from this project's own data.

**Field classification.** The design keeps a project classification when one exists and otherwise uses the paper's
14 JEL-based fields. Step 2 coded the record in JEL, so the two are reconciled: the 14 fields of Dolton and Tol
(Table B.7) are the options, and every prize work is mapped into them through its consensus JEL code (section 1).

**Outputs:** `context/00_*.md` (the nine notes, fixed upload order), `questions/fields14.json` (the Preseen question),
`results/*.csv` and `../record/tables/*.tex` (the tables behind the notes and the record). An optional note with
external 2026 signals (Kalshi, Clarivate, prizes, from `Data/convergence_2026.csv`) was built on 10 October and then
dropped on the user's decision: no information about named people enters the field question.

**Sections:** 1 the 14 fields and the mapping of the record · 2 committee coverage of the fields · 3 the notes ·
4 question draft · 5 check.

In [ ]:
import html
import json
import re
import textwrap
import xml.etree.ElementTree as ET
from collections import Counter
from fractions import Fraction
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
from IPython.display import Markdown, display

ECON = Path('/project/jevans/Dawoon/Nobel Prize/Econ')
HERE = ECON / '04_field_forecast'
CONTEXT, RESULTS, QUESTIONS, TABLES = HERE / 'context', HERE / 'results', HERE / 'questions', ECON / 'record' / 'tables'
for d in (CONTEXT, RESULTS, QUESTIONS, TABLES):
    d.mkdir(parents=True, exist_ok=True)
CFG = yaml.safe_load((ECON / 'config.yaml').read_text())
CUTOFF = CFG['today']                     # information cutoff stated in every note (config.yaml: today)
FORECAST_YEAR = 2026
ANNOUNCEMENT = 'Monday 12 October 2026, 11:45 CEST at the earliest'
pd.set_option('display.max_colwidth', 120, 'display.width', 230, 'display.max_rows', 100)

W = pd.read_csv(ECON / '02_fields/results/prize_works_jel.csv')          # 64 works, consensus JEL + each model's codes
L = pd.read_csv(ECON / 'data/econ_prizes_laureates.csv')                 # 99 laureates
L['work_id'] = L['year'].astype(str) + '-' + L['group'].astype(str)
L['share'] = L['portion'].map(lambda s: float(Fraction(s)))
# age at the award from the dates: the Nobel Prize API dates the 2022 prize 2011-10-10, so its integer age_at_award is
# eleven years too low for Bernanke, Diamond and Dybvig; an award date outside the prize year is replaced by 10 October
_born = pd.to_datetime(L['birth_date'].where(L['birth_date_precision'].eq('day'), L['birth_date'].str[:4] + '-07-01'))
_aw = pd.to_datetime(L['date_awarded'])
_aw = _aw.where(_aw.dt.year == L['year'], pd.to_datetime(L['year'].astype(str) + '-10-10'))
L['age_corr'] = (_aw - _born).dt.days / 365.2425
FAMILY = L.groupby('work_id')['family_name'].apply(lambda s: ', '.join(s))
ROSTER = yaml.safe_load((ECON / '01_committee/roster.yaml').read_text())['members']
PROFILES = {m['slug']: json.loads((ECON / '01_committee/profiles' / f"{m['slug']}.json").read_text()) for m in ROSTER}
assert len(W) == 64 and len(L) == 99 and len(PROFILES) == 11 and all(p['valid'] for p in PROFILES.values())

# JEL names at every level
NAME = {}
for node in ET.parse(ECON / 'data/classificationTree.xml').iter('classification'):
    NAME[node.findtext('code')] = html.unescape(node.findtext('description').replace('&bull;', '•')).strip()
NAME = {k: re.sub(r'\s+', ' ', v) for k, v in NAME.items()}

WRITTEN = {}


def emit(name, text):
    # Print the note exactly as it will be uploaded and write it to context/<name>.md.
    text = textwrap.dedent(text).strip() + '\n'
    path = CONTEXT / f'{name}.md'
    path.write_text(text)
    WRITTEN[name] = len(text)
    print(text)
    print(f'--- written {path.relative_to(ECON)} ({len(text):,} characters)')


def md_table(df, floatfmt='{:g}'):
    # A GitHub-style markdown table of a DataFrame (the index is included when it is named).
    d = df.reset_index() if df.index.name else df
    cells = [[floatfmt.format(v) if isinstance(v, (float, np.floating)) and not pd.isna(v) else ('' if pd.isna(v) else str(v))
              for v in row] for row in d.itertuples(index=False)]
    head = '| ' + ' | '.join(map(str, d.columns)) + ' |'
    sep = '|' + '---|' * len(d.columns)
    return '\n'.join([head, sep] + ['| ' + ' | '.join(r) + ' |' for r in cells])


def tex_escape(s):
    return str(s).replace('&', r'\&').replace('%', r'\%').replace('_', r'\_').replace('–', '--')


print(f'information cutoff {CUTOFF}; announcement {ANNOUNCEMENT}; works {len(W)}, laureates {len(L)}, profiles {len(PROFILES)}')

## 1. The 14 fields and the mapping of the prize record

Dolton and Tol (2026, Table B.7) group two-digit JEL codes into 14 fields of economics [P]. Three groups of codes
that their table leaves out are added here [D]: C8 (data collection and estimation methods) to Econometrics, M
(business economics) to Production and Industrial Organization, O5 (economywide country studies) to Development.
A work whose consensus primary code is still outside the table (D0, general microeconomics: Ostrom 2009, D02) takes
the field named most often by the three models' secondary codes, and the rule used is recorded in `field_rule`.

The history table ranks the fields by **years since the last award**, with ties broken by fewer prizes in 2016–2025
and then fewer prizes overall. This is the design's *baseline from the award record alone*; the **tier** column is
the design's operational weighting (awarded in the previous 1–2 years: large penalty; 3–5 years: penalty; a long wait:
bonus). Both are descriptions of the record, not probabilities.

In [ ]:
FIELDS = {   # field -> (JEL codes as printed in the notes, code prefixes)
    'Econometrics': ('C0-C5, C8', ['C0', 'C1', 'C2', 'C3', 'C4', 'C5', 'C8']),
    'Equilibrium, Welfare': ('C6, D5, D6', ['C6', 'D5', 'D6']),
    'Games, Market Structure': ('C7, D4', ['C7', 'D4']),
    'Behavioural, Experimental': ('C9, D9', ['C9', 'D9']),
    'Labour': ('D1, D3, I, J', ['D1', 'D3', 'I', 'J']),
    'Production, Industrial Organization': ('D2, L, M', ['D2', 'L', 'M']),
    'Public, Law, Political Economy': ('D7, H, K, P', ['D7', 'H', 'K', 'P']),
    'Information': ('D8', ['D8']),
    'Macro': ('E', ['E']),
    'Trade': ('F', ['F']),
    'Finance': ('G', ['G']),
    'Development, Economic History': ('N, O1, O2, O5', ['N', 'O1', 'O2', 'O5']),
    'Growth': ('O3, O4', ['O3', 'O4']),
    'Resources, Environment': ('Q, R', ['Q', 'R']),
}
ORDER = list(FIELDS)
SHORT = {'Econometrics': 'Econometrics', 'Equilibrium, Welfare': 'Equilibrium', 'Games, Market Structure': 'Games',
         'Behavioural, Experimental': 'Behavioural', 'Labour': 'Labour', 'Production, Industrial Organization': 'Production/IO',
         'Public, Law, Political Economy': 'Public', 'Information': 'Information', 'Macro': 'Macro', 'Trade': 'Trade',
         'Finance': 'Finance', 'Development, Economic History': 'Development/EH', 'Growth': 'Growth',
         'Resources, Environment': 'Resources'}
PREFIX = {p: f for f, (_, ps) in FIELDS.items() for p in ps}
ADDED = {'C8': 'Econometrics', 'M': 'Production, Industrial Organization', 'O5': 'Development, Economic History'}


def field_of(code):
    code = str(code or '').strip().upper()
    return PREFIX.get(code[:2]) or PREFIX.get(code[:1])


def parse_codes(s):
    m = re.match(r'\s*([A-Z]\d\d)\s*(?:\((.*)\))?\s*$', str(s))
    return (m.group(1), [c.strip() for c in (m.group(2) or '').split(',') if c.strip()]) if m else (None, [])


def assign(row):
    f = field_of(row.jel)
    if f:
        return f, 'primary code'
    votes = Counter(field_of(c) for m in 'AOG' for c in parse_codes(row[f'code_{m}'])[1] if field_of(c))
    (f, n), = votes.most_common(1)
    return f, f'secondary codes ({n} of {sum(votes.values())} mentions)'


W[['field14', 'field_rule']] = W.apply(lambda r: pd.Series(assign(r)), axis=1)
W['laureates_short'] = W['work_id'].map(FAMILY)
W['secondary_mentions'] = W.apply(lambda r: ', '.join(sorted({SHORT[f] for m in 'AOG' for c in parse_codes(r[f'code_{m}'])[1]
                                                             if (f := field_of(c)) and f != r.field14})), axis=1)
assert W.field14.notna().all()
print('mapped by a fallback rule:')
display(W.loc[W.field_rule != 'primary code', ['work_id', 'laureates_short', 'jel', 'code_A', 'code_O', 'code_G', 'field14', 'field_rule']])


def tier(since):
    if since <= 2:
        return 'large penalty (awarded 2024 or 2025)'
    if since <= 5:
        return 'penalty (awarded 2021-2023)'
    if since <= 10:
        return 'neutral (awarded 2016-2020)'
    return 'bonus: long wait (last award 2015 or earlier)'


rows = []
for f in ORDER:
    g = W[W.field14 == f]
    ys = sorted(set(g.year))
    d = np.diff(ys)
    rows.append({'field': f, 'JEL': FIELDS[f][0], 'works': len(g), 'prizes': g.share.sum(), 'award years': ', '.join(map(str, ys)),
                 'first': ys[0], 'last': ys[-1], 'years since last': FORECAST_YEAR - ys[-1],
                 'median gap': float(np.median(d)) if len(d) else np.nan,
                 'prizes 2021-2025': g[g.year >= 2021].share.sum(), 'prizes 2016-2025': g[g.year >= 2016].share.sum()})
H = pd.DataFrame(rows).set_index('field')
H['tier'] = H['years since last'].map(tier)
H = H.sort_values(['years since last', 'prizes 2016-2025', 'prizes'], ascending=[False, True, True])
H['baseline rank'] = range(1, len(H) + 1)
assert np.isclose(H.prizes.sum(), 57) and H.works.sum() == 64
display(H.drop(columns=['JEL']).style.format({'prizes': '{:.1f}', 'median gap': '{:g}', 'prizes 2021-2025': '{:.1f}', 'prizes 2016-2025': '{:.1f}'}))

H.to_csv(RESULTS / 'fields14_history.csv')
W.to_csv(RESULTS / 'prize_works_fields14.csv', index=False)
with open(TABLES / 'fields14_history.tex', 'w') as fh:
    fh.write('\\begin{tabular}{@{}rlrrlrrrr@{}}\n\\toprule\n')
    fh.write('Rank & Field (JEL) & Prizes & Last & Since & Median gap & 2021--25 & 2016--25 & Tier \\\\\n\\midrule\n')
    TIER_SHORT = {'large penalty (awarded 2024 or 2025)': 'large penalty', 'penalty (awarded 2021-2023)': 'penalty',
                  'neutral (awarded 2016-2020)': 'neutral', 'bonus: long wait (last award 2015 or earlier)': 'bonus'}
    for f, r in H.iterrows():
        gap = '--' if pd.isna(r['median gap']) else f"{r['median gap']:g}"
        fh.write(f"{r['baseline rank']} & {tex_escape(f)} ({tex_escape(r['JEL'])}) & {r['prizes']:.1f} & {r['last']} & "
                 f"{r['years since last']} & {gap} & {r['prizes 2021-2025']:.1f} & {r['prizes 2016-2025']:.1f} & {TIER_SHORT[r['tier']]} \\\\\n")
    fh.write('\\bottomrule\n\\end{tabular}\n')
print('written:', RESULTS / 'fields14_history.csv', RESULTS / 'prize_works_fields14.csv', TABLES / 'fields14_history.tex')

## 2. Committee coverage of the 14 fields

Each step-1 profile lists a member's research fields with JEL codes and the models that state the field (`support`:
A = claude-opus-5-5, O = gpt-5.5, G = gemini-3.1-pro). A member **covers** a field of the 14 when one of their
profile fields carries a code in it; ● = the profile field is stated by at least two of the three models, ○ = by one.
A member's **main field** is the field with the largest support-weighted number of codes. For comparison, Dolton and
Tol (Table A.6) assign one field and one JEL code to every committee member of 1969–2025, including the 2025
roster, which has the same eleven people as 2026 [P]. Expertise is not preference: the design (rule 7) uses this
table only as a limited adjustment.

In [ ]:
DT_A6 = {   # Dolton and Tol (2026), Table A.6, rows of the 2025 committee [P]
    'john-hassler': 'Growth (O4)', 'per-krusell': 'Growth (O4)', 'per-stromberg': 'Finance (G2)', 'peter-fredriksson': 'Labour (I2)',
    'tommy-andersson': 'Information (D8)', 'kerstin-enflo': 'Development (N1)', 'randi-hjalmarsson': 'Labour (I2)',
    'anna-dreber-almenberg': 'Behavioural (D9)', 'timo-boppart': 'Trade (F2)', 'richard-friberg': 'Labour (I2)', 'jan-teorell': 'Public (D7)'}
ROLE = {m['slug']: m['role'] for m in ROSTER}
ROLE_SHORT = {'chair': 'chair (member)', 'member': 'member', 'co-opted member': 'co-opted', 'secretary': 'secretary'}

rows, weights = [], []
for slug, r in PROFILES.items():
    for f in r['profile']['fields']:
        fs = {field_of(c) for c in f['jel_codes']} - {None}
        for x in fs:
            rows.append({'slug': slug, 'member': r['name'], 'role': ROLE[slug], 'field14': x, 'models': len(f['support']), 'profile field': f['field']})
        for c in f['jel_codes']:
            if field_of(c):
                weights.append({'slug': slug, 'field14': field_of(c), 'w': len(f['support']) / 3})
CF = pd.DataFrame(rows)
MAIN = pd.DataFrame(weights).groupby(['slug', 'field14'])['w'].sum().unstack().idxmax(axis=1)
M = CF.groupby(['member', 'field14'])['models'].max().unstack().reindex(columns=ORDER)
M = M.reindex([PROFILES[m['slug']]['name'] for m in ROSTER])          # roster order: chair, members, co-opted, secretary
MARK = M.apply(lambda col: col.map(lambda v: '●' if v >= 2 else ('○' if v == 1 else '')))
MARK.columns = [SHORT[c] for c in MARK.columns]
MARK.insert(0, 'role', [ROLE[m['slug']] for m in ROSTER])
display(MARK.style.set_caption('● field stated by 2-3 models, ○ by one model'))

COV = pd.DataFrame({'members (any)': (M >= 1).sum(), 'members (2+ models)': (M >= 2).sum()})
role_of = pd.Series({PROFILES[m['slug']]['name']: ROLE[m['slug']] for m in ROSTER})
for role, label in (('member', 'regular members incl. chair'), ('co-opted member', 'co-opted members'), ('secretary', 'secretary')):
    names = role_of.index[role_of.isin([role, 'chair'] if role == 'member' else [role])]
    COV[label] = (M.loc[names] >= 1).sum()
COV['main field of'] = [', '.join(PROFILES[s]['name'].split()[-1] for s, f in MAIN.items() if f == c) for c in COV.index]
COV = COV.join(H[['prizes', 'last']]).rename(columns={'prizes': 'prizes 1969-2025', 'last': 'last award'})
COV.index.name = 'field'
display(COV)

PER_MEMBER = pd.DataFrame({
    'role': role_of,
    'main field (profiles)': [SHORT[MAIN[m['slug']]] for m in ROSTER],
    'fields covered (2+ models)': [', '.join(SHORT[c] for c in ORDER if M.loc[PROFILES[m['slug']]['name'], c] >= 2) for m in ROSTER],
    'Dolton-Tol A.6': [DT_A6[m['slug']] for m in ROSTER]}, index=role_of.index)
display(PER_MEMBER)

CF.to_csv(RESULTS / 'committee_fields14_long.csv', index=False)
COV.to_csv(RESULTS / 'committee_fields14_coverage.csv')
PER_MEMBER.to_csv(RESULTS / 'committee_members_fields14.csv')
ABBR = {'Econometrics': 'Em', 'Equilibrium': 'Eq', 'Games': 'Gm', 'Behavioural': 'Bh', 'Labour': 'Lb', 'Production/IO': 'IO',
        'Public': 'Pb', 'Information': 'In', 'Macro': 'Ma', 'Trade': 'Tr', 'Finance': 'Fi', 'Development/EH': 'Dv', 'Growth': 'Gr', 'Resources': 'Re'}
with open(TABLES / 'committee_fields14.tex', 'w') as fh:
    fh.write('\\begin{tabular}{@{}ll' + 'c' * 14 + 'l@{}}\n\\toprule\n')
    fh.write('Member & Role & ' + ' & '.join(ABBR[SHORT[c]] for c in ORDER) + ' & Dolton--Tol A.6 \\\\\n\\midrule\n')
    for m in ROSTER:
        name = PROFILES[m['slug']]['name']
        marks = ' & '.join({'●': r'$\bullet$', '○': r'$\circ$', '': ''}[MARK.loc[name, SHORT[c]]] for c in ORDER)
        fh.write(f"{tex_escape(name)} & {ROLE_SHORT[ROLE[m['slug']]]} & {marks} & {tex_escape(DT_A6[m['slug']])} \\\\\n")
    fh.write('\\midrule\n' + 'members covering & & ' + ' & '.join(str(int(COV.loc[c, 'members (any)'])) for c in ORDER) + ' & \\\\\n')
    fh.write('\\bottomrule\n\\end{tabular}\n')
print('written:', *(RESULTS / f for f in ('committee_fields14_long.csv', 'committee_fields14_coverage.csv', 'committee_members_fields14.csv')), TABLES / 'committee_fields14.tex')

## 3. The context notes

One cell per note. The nine files are uploaded in name order (`add-context` sorts them); the text each cell prints
is the note. All nine are attached with the `assume_true` treatment: 00_1 is the brief, 00_2 and 00_3 are the
definitions the resolution depends on, 00_4 and 00_5 are the record, 00_6 to 00_9 are the rest of the design.

### 00_1 Instruction: goal, evidence tags, rotation prior, operational weights, required output

In [ ]:
emit('00_1_instruction', f"""
# Instruction: forecasting the field of the 2026 Sveriges Riksbank Prize in Economic Sciences

Information cutoff: {CUTOFF}, before the 2026 announcement ({ANNOUNCEMENT}).

## Goal and evidence tags
Forecast which field of economics the 2026 prize will recognize. This stage is about the field only: the ranking of
people and the composition of a shared prize are later stages. Forecasting the field first is a way of organizing
the analysis; it does not assert that the committee is known to decide the field before the people. [P, M, D]
Every statement in these notes carries a tag: [P] a result reported by Dolton and Tol (2026), [M] an opinion raised
at the project meeting of 9 October 2026, [D] a design choice of this forecast, [O] information confirmed from
official sources (nobelprize.org), [S] a number computed from this project's own data (the 1969-2025 record coded in
JEL by three language models; the profiles of the 2026 committee). The notes attached to this question are the
reference material for rules 2, 3, 4 and 7; the maturity assessment of rule 5 and the societal reading of rule 6 draw
on your own knowledge, which you mark as such.

## 1. Field rotation is the primary prior [D]
Let the rotation of past fields, the concentration of recent awards and the time since each field's last award
determine the baseline ranking. First build the baseline from the award record alone (note 00_4), then adjust it for
the accumulation and maturity of work that has not been awarded (rule 5). Societal attention and committee expertise
adjust this judgment only in a supporting role. Do not overturn the rotation baseline on the strength of recent news
or of one famous candidate. If a recently awarded field stays in the final top group, state the concrete work-based
grounds for the exception. The strong rotation assumption does not mean an automatic probability of zero for any field
or a fixed turn order.

## 2. One classification for the record and for this year's options [P, D]
The options are the 14 fields of Dolton and Tol, defined as groups of JEL codes; note 00_2 gives the definitions
and the boundary rules for overlapping areas (growth / development and economic history / macro; information / games
and market structure; institutions), and note 00_3 maps every past prize work into the fields. Societal themes (AI, tariffs,
migration, war, democracy) are linked to a field and to concrete work; broad fields and narrow applied topics are
not mixed as options of the same level. Do not reclassify the same work under a new field name to escape the
recent-award penalty.

## 3. Strong priority to the award record [P, D]
For each field use its last award year, the years elapsed to 2026, the number of prizes in the last 5 and 10 years
and the core work already recognized (notes 00_3, 00_4). Look at the whole record, and apply these operational
weights: a field awarded in the previous 1-2 years (2024, 2025): large penalty, since the opportunity for
recognition is likely to move elsewhere; a field awarded 3-5 years ago (2021-2023): penalty, lower still when the
field was recognized several times recently; a field that has not been awarded for a long time: strong bonus,
especially when a long wait and sufficient unawarded work coincide. Record the cases in which the same field was
chosen in consecutive years and the exceptions; do not set a field to zero because a rare transition has never
occurred. The 1-2 / 3-5 year tiers and the strength of the penalties are operational settings of this forecast, not
thresholds estimated by the paper. The paper reports positive effects of the years since a field's last award and of
the number of candidates in the field, but the dummy for having won the previous year is not significant in its main
model (note 00_9); the strong recency penalty is therefore a deliberate rotation assumption, not a replicated rule.
Waiting time, recency and transition patterns are related descriptions of the same record: do not count one fact as
several independent pieces of evidence.

## 4. Use the awards through 2025 exactly [O, D] -- see note 00_5.
## 5. Is there enough unawarded, mature work? [P, M, D] -- see note 00_6.
## 6. Societal context is auxiliary [M, D] -- see note 00_7.
## 7. Committee expertise and change over time are limited adjustments [P, O, D] -- see note 00_8.
## 8. Do not copy the paper's numbers into this year's probabilities [P, D] -- see note 00_9.
Use only information dated before the cutoff. Where candidate counts, member expertise or societal attention are not
verified, say so.

## 9. Required output [D]
For each of the 14 fields, concisely: (1) the field and the representative contributions that could be recognized;
(2) last award year, waiting time, prizes in the last 5 and 10 years; (3) the baseline rank from the record alone and
the rotation advantages or disadvantages; (4) the accumulation and maturity of unawarded work and how it differs from
the work already awarded; (5) how much societal context and committee information change the baseline; (6) the final
rank and probability with the main supporting and opposing grounds. A field that rises far above its baseline rank
gets a concrete reason; a recently awarded field that stays in the top group gets work-based grounds beyond
topicality. Define the probability event first: this question asks for one representative field, so the 14 options
are mutually exclusive under the classification of note 00_2 and the question's resolution rule, and the probabilities
sum to 100 % (do not mix in a uniform floor). The question resolves by the field of the primary JEL code of the 2026
motivation, coded by the same three-model procedure as the record of note 00_3 and mapped by note 00_2; a prize divided
between fields resolves by its overall motivation when the announcement gives one, otherwise by the larger share, then
by the contribution named first. The keywords in the option labels illustrate the fields; the JEL code groups define
them. Report the baseline distribution from the record alone, the distribution
after the maturity adjustment, and the submitted distribution after the auxiliary adjustments, with every adjustment
and its factor listed by field. The main result is the one with the rotation prior applied strongly. The aim is to
identify the fields that have gone unrecognized for a long time and have enough mature work to recognize now.
""")

### 00_2 Field classification: the 14 fields and the boundary rules

In [ ]:
added = '; '.join(f'{k} to {v}' for k, v in ADDED.items())

emit('00_2_field_classification', f"""
# Field classification: the 14 fields and the boundary rules

Information cutoff: {CUTOFF}.

## The options [P, D]
The 14 fields are those of Dolton and Tol (2026, Table B.7), each a group of two-digit JEL codes; they are not the
JEL letters. Codes the paper's table leaves out are added so that every code of the awarded record has a field [D]:
{added}. A work whose consensus primary code is still outside the table (D0, general microeconomics) takes the field
named most often by the three models' secondary codes.

| Field | JEL codes |
|---|---|
""" + '\n'.join(f'| {f} | {FIELDS[f][0]} |' for f in ORDER) + f"""

## Boundary rules, fixed before the analysis [D]
- Growth (O3-O4) covers innovation, technological change, productivity and growth theory, including the growth
  effects of institutions (O43). Development and economic history (N, O1-O2) covers development economics, poverty
  and the history of economies. Macro (E) covers business cycles, money, consumption and saving, fiscal and monetary
  policy. A contribution is placed by the core of what a motivation would recognize, as the step-2 coding did.
- Information (D8) covers asymmetric information, contracts, mechanism design and the economics of information;
  games and market structure (C7, D4) covers game theory, bargaining, matching and auction design and market
  structure. Mechanism design counts as Information, auction and matching design as Games.
- Institutions: the 2024 prize (institutions and prosperity, O43) is Growth under this mapping; its secondary codes
  (P16, N40, D02) show the overlap with Public, law and political economy and with Development and economic history.
  A 2026 award for the political economy of institutions and prosperity would overlap in substance with 2024 and
  carries the 2024 penalty; a Public, law and political economy award for distinct work (taxation, public choice,
  law and economics, political economy of policy) does not.
- Labour (D1, D3, I, J) includes household economics, distribution, health and education economics. Resources and
  environment (Q, R) includes urban and regional economics.
- The same work is never re-labelled to a different field to avoid a recency penalty.

The record of 1969-2025 mapped into these fields is in note 00_3; the waiting times and the baseline rank in note 00_4.
""")

### 00_3 The record mapped into the fields: every prize work 1969–2025

In [ ]:
lines = []
for f in ORDER:
    g = W[W.field14 == f].sort_values(['year', 'group'])
    lines.append(f"\n**{f}** (JEL {FIELDS[f][0]}): {len(g)} work{'s' if len(g) > 1 else ''}, {g.share.sum():.1f} prizes, last {g.year.max()}")
    for r in g.itertuples():
        extra = f'; field by {r.field_rule}' if r.field_rule != 'primary code' else ''
        sec = f'; secondary codes also point to {r.secondary_mentions}' if r.secondary_mentions else ''
        share = '' if r.share == 1 else f', {Fraction(r.share).limit_denominator(4)} of the prize'
        lines.append(f"- {r.year} {r.laureates_short}{share} [{r.jel} {NAME.get(r.jel, '')}{extra}{sec}]")
works_by_field = '\n'.join(lines)

emit('00_3_prize_works_by_field', f"""
# The record mapped into the 14 fields: every prize work 1969-2025 [S, O]

Information cutoff: {CUTOFF}. One work = one motivation of a prize year (57 prizes, 7 of them divided between two
works, 64 works); "1/2 of the prize" marks the share of a divided prize. Field = the field of note 00_2 that contains
the work's consensus JEL code, given from the official motivation by three language models (claude-opus-5-5, gpt-5.5,
gemini-3.1-pro; all three agree on the JEL letter for 58 of 64 works). The code and its JEL name follow in brackets,
with the fallback rule where the primary code lies outside the 14 fields, and the other fields the models' secondary
codes point to. The official motivations are at nobelprize.org; those of 2021-2025 are quoted in note 00_5.
{works_by_field}
""")

### 00_4 Prize history by field: waiting times, recent frequency, baseline rank, sequence facts

In [ ]:
T = H.reset_index()[['baseline rank', 'field', 'prizes', 'award years', 'last', 'years since last', 'median gap',
                     'prizes 2021-2025', 'prizes 2016-2025', 'tier']].copy()
T.columns = ['baseline rank', 'field', 'prizes 1969-2025', 'award years', 'last award', 'years since last', 'median gap between awards',
             'prizes 2021-2025', 'prizes 2016-2025', 'operational tier']
T['prizes 1969-2025'] = T['prizes 1969-2025'].map('{:.1f}'.format)
T['prizes 2021-2025'] = T['prizes 2021-2025'].map('{:.1f}'.format)
T['prizes 2016-2025'] = T['prizes 2016-2025'].map('{:.1f}'.format)
T['median gap between awards'] = T['median gap between awards'].map(lambda v: '' if pd.isna(v) else f'{v:g}')
seq = W.groupby('year')['field14'].apply(lambda s: ' + '.join(SHORT[x] for x in sorted(set(s))))
last_ten = '; '.join(f'{y} {v}' for y, v in seq.tail(10).items())
same = [y for y in seq.index[1:] if set(W[W.year == y].field14) & set(W[W.year == y - 1].field14)]
repeat_years = ', '.join(f"{y} ({'/'.join(SHORT[x] for x in sorted(set(W[W.year == y].field14) & set(W[W.year == y - 1].field14)))})" for y in same)
def n_models_mentioning(row, field):
    return sum(any(field_of(c) == field for c in parse_codes(row[f'code_{m}'])[1]) for m in 'AOG')


macro_secondary = sorted(set(W[(W.field14 != 'Macro') & (W.apply(lambda r: n_models_mentioning(r, 'Macro'), axis=1) >= 2)].year))

emit('00_4_prize_history_by_field', f"""
# Prize history by field, 1969-2025 [S]

Information cutoff: {CUTOFF}. Prizes are share-weighted (a prize divided between two works counts 1/2 for each);
the 57 prizes and 64 works are classified as in note 00_2. "Years since last" is measured to 2026. The baseline
rank orders the fields by years since the last award, ties broken by fewer prizes in 2016-2025 and then fewer prizes
overall: it is the design's baseline from the record alone (rule 3), not a probability. The operational tier is the
design's weighting rule; it does not set any field to zero.

{md_table(T)}

## Sequence facts [S]
- Fields of the last ten prizes: {last_ten}.
- Years in which a field of the prize was also a field of the previous year's prize: {repeat_years}. Across
  1969-2025 that is {len(same)} of 56 transitions. At the level of JEL letters the step-2 analysis found 10 such
  repeats against 10.2 expected if the order were random (P = 0.56), so the record alone shows neither avoidance
  nor favouring of the previous year's field; the strong rotation prior is a design choice (rule 1).
- Growth followed itself in 2024 -> 2025 (institutions and prosperity, then creative destruction); it is the only
  case since 1969 in which the same two-digit JEL field (O4) won in consecutive years.
- Macro was the main field eight times between 1974 and 2006 and not since (20 years against a median gap of 3),
  but at least two of the three models give a macroeconomic (E) secondary code to the works of
  {', '.join(map(str, macro_secondary))} (econometric modelling of fluctuations, empirical macro, financial crises),
  so macroeconomic work has been recognized under other headings.
- General equilibrium and welfare theory (last 1998) and the econometric and mathematical modelling of the 1970s
  (C5, C6: last 1980 and 1975) have not returned at the two-digit level; growth, development, labour and empirical
  work have gained since 2015.
- Laureates per prize rose from 1.55 (1969-79) to 2.5 (2020-25); two of the last five prizes were divided between
  two works (2021, 2025), so a 2026 prize may recognize two fields; the question's rule for that case is in its
  resolution criteria.
""")

### 00_5 The 2021–2025 prizes: official motivations, codes, field and the weight they carry

In [ ]:
recent = W[W.year >= 2021].sort_values(['year', 'group'])
blocks = []
for r in recent.itertuples():
    laur = L[L.work_id == r.work_id].sort_values('share', ascending=False)
    people = '; '.join(f"{x.laureate} ({x.portion}, {x.affiliations_short.split(';')[0].strip() if isinstance(x.affiliations_short, str) else '-'}, age {int(round(x.age_corr))})"
                       for x in laur.itertuples())
    codes = ' / '.join(f"{m}: {r.__getattribute__('code_' + m)}" for m in 'AOG')
    blocks.append(f"""- **{r.year}** {people}: "{r.motivation}" [O]. Consensus code {r.jel} ({NAME.get(r.jel, '')}); the models' codes
  {codes} [S]. Field: **{r.field14}**{' (secondary codes also point to ' + r.secondary_mentions + ')' if r.secondary_mentions else ''}.
  Weight in 2026: {tier(FORECAST_YEAR - r.year)}.""")
recent_blocks = '\n'.join(blocks)

emit('00_5_recent_prizes_2021_2025', f"""
# The prizes of 2021-2025 and the weight they carry in 2026 [O, S, D]

Information cutoff: {CUTOFF}. The 2026 forecast reflects the awards through 2025 exactly; these five years are
examples with the weights of rule 3, they do not replace the whole record of note 00_4.

{recent_blocks}

## How to read them [D]
- 2025 (Mokyr; Aghion and Howitt: innovation-based growth) and 2024 (Acemoglu, Johnson, Robinson: institutions and
  prosperity) carry the large previous-years penalty. Under note 00_2 both the 2024 and the 2025 Aghion-Howitt works
  are Growth, and the 2025 Mokyr half is Development and economic history: Growth has 1.5 prizes in 2024-2025 and
  Development and economic history 0.5, so both are penalized, Growth more.
- 2023 (Goldin: women's labour market outcomes) counts as the 3-5 year tier for Labour; Labour also holds half of
  2021 (Card), so it is the most frequently recognized field of 2021-2025 (1.5 prizes) and is weighted down for
  concentration as well. 2022 Finance (banks and financial crises) and the 2021 half for Econometrics (Angrist and
  Imbens, causal inference) are in the 3-5 year tier.
- "Technology" is not one field: research on AI, automation and employment is Labour or Production and industrial
  organization by the core of the contribution, not the 2025 growth work, and is not penalized for sharing a word
  with it. Conversely, a contribution that in substance overlaps a recent award (endogenous growth and creative
  destruction; the institutional origins of prosperity; the gender gap in labour markets) is not re-packaged under
  a new label to avoid the penalty.
- The 2024 "institutions" work is mapped as Growth; before judging Public, law and political economy and Development
  and economic history, apply the boundary rules of note 00_2 consistently.
""")

### 00_6 Unawarded, mature work: what to assess and what the data do not provide

In [ ]:
B7 = {'Econometrics': (33, 13), 'Equilibrium, Welfare': (21, 8), 'Games, Market Structure': (20, 7), 'Behavioural, Experimental': (12, 3),
      'Labour': (27, 4), 'Production, Industrial Organization': (23, 3), 'Public, Law, Political Economy': (21, 3), 'Information': (23, 13),
      'Macro': (39, 12), 'Trade': (22, 4), 'Finance': (21, 11), 'Development, Economic History': (23, 7), 'Growth': (23, 9),
      'Resources, Environment': (10, 2)}   # Dolton and Tol (2026) Table B.7: candidates (their constructed pool, 1969-2025), laureates [P]
b7 = pd.DataFrame([{'field': f, 'candidates in the paper\'s pool (1969-2025)': c, 'laureates in the paper': l,
                    'prizes in this project (share-weighted)': f'{H.loc[f, "prizes"]:.1f}'} for f, (c, l) in B7.items()])

emit('00_6_unawarded_maturity', f"""
# Is there enough unawarded, mature work in the field? [P, M, D]

Information cutoff: {CUTOFF}.

## What to assess [M, D]
After the baseline from the record (note 00_4), judge for each field how much mature work remains unawarded and
whether living researchers represent it. Consider: defining works that newly formulated or solved an important
problem; research programmes accumulated over a long period; settlement in textbooks and syllabi; diffusion into
follow-on research in and outside the field. Account for what changed after the 2025 prize: contributions that have
now been recognized, deaths, and work and people that have newly matured. These are the meeting's suggestions used
as a qualitative assessment; they are not a composite indicator the paper validated. Count a body of work once,
however many co-contributors it has, and do not use the total number of papers or of researchers in a field as its
size. Give high priority where a long wait and an important contribution clearly distinct from already-awarded work
coincide; a long wait alone is not enough.

## What the paper's pool sizes mean [P]
Dolton and Tol count "candidates" per field in a pool they constructed for 1969-2025 (winners of major prizes and
similar signals), not nomination lists. The number of candidates in a field is the strongest predictor of the field
winning in their model (note 00_9). The pool counts below are historical totals: they cannot be used as the number of
living, unawarded candidates in 2026, and this project has not built a 2026 candidate list for the field stage.

{md_table(b7)}

## What this project does not provide [D]
No candidate lists, citation counts or textbook measures per field are attached. Your assessment of maturity draws on
your own knowledge of the literature up to the cutoff; mark it as your assessment, and mark uncertain claims about
who is alive or unawarded as unverified.
""")

### 00_7 Societal context: auxiliary information only

In [ ]:
emit('00_7_societal_context', f"""
# Societal context: auxiliary to the rotation judgment [M, D]

Information cutoff: {CUTOFF}.

The project meeting of 9 October 2026 raised these themes for review: AI and jobs; trade and tariffs; war and
migration; inequality; institutions and democracy. The list is not a ranking of candidate fields and it is not a
measurement of current attention.

For each theme ask: how it connects to long-accumulated contributions of economics (which field, which body of
work); whether it is internationally important rather than local; and whether the attention has had time to turn
into an award. A lag between societal events and awards may exist; do not apply "1-2 years" as a fixed law.

Dolton and Tol did not test news volume or societal attention as determinants of the prize; there is no empirical
estimate to lean on. Topicality alone does not offset the large penalty of a recently awarded field. A field with
little public attention but a long accumulation of theory, methods or foundational results stays a strong
contender. Do not assume that the committee intends a political message; treat such a motive as possible, not as
given.
""")

### 00_8 The 2026 committee: roster, roles and field coverage from the step-1 profiles

In [ ]:
def short_topic(t, n=80):
    t = re.split(r':|, including| including|\(', t)[0].strip().rstrip('.,;')
    return t if len(t) <= n else t[:n].rsplit(' ', 1)[0] + '...'


member_lines = []
for m in ROSTER:
    p = PROFILES[m['slug']]
    name = p['name']
    covered = [(SHORT[c], M.loc[name, c]) for c in ORDER if M.loc[name, c] >= 1]
    cov_txt = ', '.join(f"{c}{'' if v >= 2 else ' (one model)'}" for c, v in covered)
    interests = sorted(p['profile']['recent_interests'], key=lambda i: -len(i['support']))[:3]
    int_txt = '; '.join(short_topic(i['topic']) for i in interests)
    member_lines.append(f"- **{name}** ({m['role']}; {m['affiliation']}). Main field by the profile: "
                        f"{SHORT[MAIN[m['slug']]]}; fields covered: {cov_txt}. Recent interests: {int_txt}. Dolton-Tol A.6: {DT_A6[m['slug']]}.")
members_txt = '\n'.join(member_lines)
cov = COV.reset_index()[['field', 'members (any)', 'members (2+ models)', 'regular members incl. chair', 'co-opted members', 'secretary', 'main field of', 'prizes 1969-2025', 'last award']].copy()
cov['prizes 1969-2025'] = cov['prizes 1969-2025'].map('{:.1f}'.format)

emit('00_8_committee_2026', f"""
# The 2026 committee and its fields [O, S, P, D]

Information cutoff: {CUTOFF}.

## Roster [O]
Committee for the Prize in Economic Sciences in Memory of Alfred Nobel 2026 (nobelprize.org, 9 October 2026).
Regular members: Tommy Andersson, Anna Dreber Almenberg, Peter Fredriksson, John Hassler (chair), Per Strömberg.
Co-opted members: Timo Boppart, Kerstin Enflo, Richard Friberg, Randi Hjalmarsson, Jan Teorell. Secretary: Per
Krusell. Co-opted members are appointed for the year to add expertise; the secretary runs the committee's work. Do
not assume one vote per person or any voting rule.

## Field coverage from the profiles [S]
Each member's research fields, with JEL codes, were described from their own knowledge by three language models
(claude-opus-5-5, gpt-5.5, gemini-3.1-pro; no web search) and merged; a member covers a field of note 00_2 when one of
their profile fields has a code in it. "2+ models" = the profile field is stated by at least two models. The last
two columns set the coverage beside the prize record.

{md_table(cov)}

## Members [S, P]
{members_txt}

## How to use this [P, D]
- Expertise is not preference, acquaintance, lobbying or a settled probability for the field. A field no member
  covers (Trade: one member, through pricing-to-market and exchange-rate pass-through) can still be awarded; the
  committee consults external referees.
- Distinguish regular members, co-opted members and the secretary; do not give them arbitrary equal weights.
- In Dolton and Tol, proximity between the committee's fields and a candidate's field is not significant for the
  field (Table 2) and is significant at the individual level only in the 1998-2025 subsample (Table E.12). Do not
  transfer that into a strong causal effect at the field level; use coverage as a limited adjustment and say by how
  much.
- Field structure, candidate pools and the committee have changed over the decades; compare periods, but do not
  treat a pattern from a small recent sample as a confirmed new rule.
- The profiles stop at the models' training data (roughly 2024) and can miss recent work; the comparison with the
  paper's one-field assignment (last item of each member line) shows where the sources differ (Boppart: growth and
  structural change in the profiles, Trade in the paper; Friberg: industrial organization and international pricing
  in the profiles, Labour in the paper).
""")

### 00_9 What Dolton and Tol (2026) report, and why its numbers are not this year's probabilities

In [ ]:
emit('00_9_paper_evidence_and_limits', f"""
# What the paper reports, and the limits of using it [P, D]

Information cutoff: {CUTOFF}. Source: Peter J. Dolton and Richard S. J. Tol, "The Process and Dynamics of the Nobel
Memorial Prize in Economics, 1969-2025", arXiv:2603.20767v1, 21 March 2026 (Table 2 p. 34, Table B.7 p. 50, Table
E.12 pp. 72-73, pp. 16-21). Figures below were read from the paper on 10 October 2026.

## Findings on the field [P]
- Field-level logit, 14 fields x 57 years = 798 field-years. With the first-order transition matrix included
  (estimated separately for 1969-1994, when Lindbeck was on the committee, and 1995-2025), the consolidated model
  keeps: the number of candidates in the field (19.8, p < 0.001), the transition probability (75.8, p < 0.001), the
  years since the field last won (0.058, p < 0.01) and a time trend (0.025, p < 0.05). Not significant: citations to
  the most cited paper in the field, total citations, proximity to the committee, the number of previous prizes in
  the field, whether the field has never won, whether the field won in the previous year (0.07 with the matrix,
  -1.8 without, both insignificant), publications in the last five years.
- Without the transition matrix the number of candidates is the only significant variable; the log-likelihood falls
  from -137 to -198 and the pseudo-R2 from 37 % to 9.1 %.
- The authors read the result as "semi-regular rotation" whose pattern changed after Lindbeck left; larger fields
  (by their candidate pool) win more often; a field kept waiting becomes more likely.
- Individual level (Table 3, E.12): the probability of winning peaks at about 70-71 years of age; having a student
  or a co-author who won matters, broader networks do not; the committee's thematic proximity to a candidate is
  significant only in the 1998-2025 split; citation indicators were significant before 1997 and not after.

## Why these numbers are not 2026 probabilities [P, D]
- The fit is in-sample: it explains 1969-2025. The transition matrix is estimated on the whole period (the window
  that includes the award year itself fits best, which is description, not prediction). Using data through 2025
  for 2026 is right; quoting the fit as validated forecasting skill is not.
- 14 fields give 196 transition cells but there are only 56 consecutive-year transitions; most cells are empty and
  individual paths are sparse (note 00_4 lists the repeats).
- Do not convert coefficients, pseudo-R2, un-normalized transition scores or historical "who should have won" scores
  into 2026 probabilities, and do not move the individual-level regularities (peak age about 71, years since the
  PhD, lineage or co-authorship with laureates) into fixed points for fields.
- Citation results depend on specification, period and pool; do not generalize that citations are meaningless.
- The paper's candidate counts are historical totals over its constructed pool (note 00_6), not the number of
  living, unawarded candidates in 2026.
""")

## 4. Draft of the Preseen question

A multiple-choice question over the 14 fields, mutually exclusive and comprehensive under the classification of note
00_2 (the resolution rule covers divided prizes and codes outside the 14 fields). Written to `questions/fields14.json`
in the format `nobel_preseen_exp.py create --question` reads; review before creating the arms.

In [ ]:
KEYWORDS = {   # three or four illustrative keywords per field, consistent with the boundary rules of note 00_2
    'Econometrics': 'econometric theory and methods, causal inference, time series, data and estimation methods',
    'Equilibrium, Welfare': 'general equilibrium, welfare economics, social choice, mathematical and optimization methods',
    'Games, Market Structure': 'game theory, bargaining, matching, auctions and market design',
    'Behavioural, Experimental': 'behavioural economics, psychological and intertemporal decision making, laboratory and field experiments',
    'Labour': 'labour markets, household and family economics, income distribution, health and education economics',
    'Production, Industrial Organization': 'theory of the firm, transaction costs and organizations, industrial organization and regulation, business economics',
    'Public, Law, Political Economy': 'public economics and taxation, collective decision making and public choice, law and economics, political economy and economic systems',
    'Information': 'asymmetric information, contract theory, mechanism design, decision making under uncertainty',
    'Macro': 'business cycles, money and monetary policy, consumption and saving, fiscal policy, national accounts',
    'Trade': 'international trade, international finance and open-economy macroeconomics, exchange rates',
    'Finance': 'asset pricing, corporate finance, banking and financial crises, financial markets',
    'Development, Economic History': 'development economics, poverty, economic history, economywide country studies',
    'Growth': 'economic growth, innovation and technological change, productivity, institutions and growth',
    'Resources, Environment': 'environmental and climate economics, natural resources and energy, agriculture, urban and regional economics',
}
question = {
    'type': 'multiple_choice',
    'title': 'Which field of economics will the 2026 Sveriges Riksbank Prize in Economic Sciences recognize?',
    'description': (f'The 2026 Sveriges Riksbank Prize in Economic Sciences in Memory of Alfred Nobel is scheduled to be announced on '
                    f'{ANNOUNCEMENT}. The options are the 14 fields of economics of Dolton and Tol (2026), each defined as a group of JEL '
                    'codes (in brackets) and illustrated by a few keywords. The field of the awarded contribution is read from the official '
                    'motivation through its JEL code; exactly one option resolves.'),
    'resolution_criteria': (
        'Resolves to the field that contains the primary JEL code of the contribution recognized by the official motivation of the 2026 prize '
        '(nobelprize.org). The primary code is the level-3 JEL code of the core of the recognized contribution, determined by the same '
        'procedure that coded the prizes of 1969-2025: three language models code the official motivation independently, and their consensus '
        '(JEL letter, then two-digit field, then code; a two-model majority, otherwise the highest score) is mapped to the fields by the code '
        'groups in the option labels. If the prize is divided between contributions in different fields, the overall motivation of the prize is '
        'coded when the announcement gives one; otherwise the contribution with the larger share counts; if the shares are equal, the '
        'contribution named first in the announcement counts. If the primary code is in no option (JEL A, B, Y, Z or a general code such as '
        'D0), the field named by the majority of the contribution\'s secondary codes counts; if no field can be assigned, the question is '
        'annulled.'),
    'fine_print': ('Classification conventions: growth effects of institutions (O43) are Growth; mechanism design (D82) is Information; auction '
                   'and matching design (D44, C78) is Games and market structure; household, health and education economics are Labour; urban '
                   'and regional economics are Resources and environment; business economics (M) is Production and industrial organization; '
                   'economywide country studies (O5) are Development and economic history. The keywords in the option labels are '
                   'illustrative; the JEL code groups define the fields.'),
    'options': [f'{f} (JEL {FIELDS[f][0]}): {KEYWORDS[f]}' for f in ORDER],
    'options_are_mutually_exclusive': True,
    'options_are_comprehensive': True,
}
(QUESTIONS / 'fields14.json').write_text(json.dumps(question, indent=2, ensure_ascii=False) + '\n')
print(json.dumps(question, indent=2, ensure_ascii=False))
print('written:', QUESTIONS / 'fields14.json')

## 5. Check

Sizes of the notes (the earlier Preseen notes of this project were 1,200–4,500 characters; the longest cards about
6,000) and the submit command. Nothing is uploaded by this notebook; `run_field.sh` does that.

In [ ]:
sizes = pd.Series(WRITTEN, name='characters').to_frame()
sizes['words'] = [len((CONTEXT / f'{n}.md').read_text().split()) for n in sizes.index]
sizes['upload order'] = range(1, len(sizes) + 1)
assert all(n.startswith('00_') for n in sizes.index) and len(sizes) == 9, 'context/ must hold exactly the nine 00_ notes'
display(sizes)
big = sizes[sizes.characters > 9000]
print('notes above 9,000 characters (consider splitting):', ', '.join(big.index) if len(big) else 'none')
print(f'total {sizes.characters.sum():,} characters in {len(sizes)} notes')
print(textwrap.dedent(f"""
Submit (from this folder, PRESEEN_API_KEY in the environment): bash run_field.sh nobel26-econ-fields
  = create the two private arms (control, main) from {QUESTIONS.name}/fields14.json, attach the nine notes of
    {CONTEXT.name}/ to main as assume_true, run one rep per arm, poll, write preseen_exp/fields14/runs.csv.
"""))